In [1]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
PRED = OUT + '/predictions'
import os, sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
os.makedirs(PRED, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
assert torch.cuda.is_available(), 'Switch runtime to GPU'
dev = 'cuda'
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def load(f):
    return ((torch.tensor(np.load(f'{DATA}/data/{f}')).permute(0, 3, 1, 2).float().div(255) - mean) / std).to(dev)
X, XT = load('train_images.npy'), load('test_images.npy')
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = torch.tensor(np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)).to(dev)
va = torch.tensor(np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)).to(dev)
clean, noisy = torch.tensor(lab.clean_label.values).to(dev), torch.tensor(lab.noisy_label.values).to(dev)
Xtr, Xva, ytr = X[tr], X[va], clean[tr]
yva_n, yva_c = noisy[va], clean[va]
del X

def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def predict(T):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([net(T[k:k + 1000]).float().argmax(1) for k in range(0, len(T), 1000)])

Mounted at /content/drive


In [2]:
torch.manual_seed(0); np.random.seed(0)
E, B = 60, 128
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            loss = F.cross_entropy(net(augment(Xtr[b])), ytr[b])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    pv = predict(Xva)
    print(ep + 1, f'val_noisy {(pv == yva_n).float().mean().item():.4f}', f'val_clean {(pv == yva_c).float().mean().item():.4f}')
pd.DataFrame({'id': np.arange(len(XT)), 'pred': predict(XT).cpu().numpy()}).to_csv(f'{PRED}/t1_clean_ref.csv', index=False)
print('saved')

1 val_noisy 0.3162 val_clean 0.4284
2 val_noisy 0.3960 val_clean 0.5468
3 val_noisy 0.4106 val_clean 0.5890
4 val_noisy 0.4450 val_clean 0.6464
5 val_noisy 0.4436 val_clean 0.6556
6 val_noisy 0.4922 val_clean 0.7340
7 val_noisy 0.5036 val_clean 0.7246
8 val_noisy 0.5292 val_clean 0.7990
9 val_noisy 0.5294 val_clean 0.8040
10 val_noisy 0.5184 val_clean 0.7800
11 val_noisy 0.4868 val_clean 0.7142
12 val_noisy 0.5354 val_clean 0.7790
13 val_noisy 0.5244 val_clean 0.7802
14 val_noisy 0.5232 val_clean 0.7856
15 val_noisy 0.5410 val_clean 0.7934
16 val_noisy 0.5360 val_clean 0.8138
17 val_noisy 0.5558 val_clean 0.8506
18 val_noisy 0.5312 val_clean 0.7942
19 val_noisy 0.5590 val_clean 0.8530
20 val_noisy 0.5488 val_clean 0.8332
21 val_noisy 0.5280 val_clean 0.8104
22 val_noisy 0.5606 val_clean 0.8634
23 val_noisy 0.5484 val_clean 0.8444
24 val_noisy 0.5616 val_clean 0.8588
25 val_noisy 0.5652 val_clean 0.8680
26 val_noisy 0.5680 val_clean 0.8696
27 val_noisy 0.5484 val_clean 0.8258
28 val_noi